<table border=1 width="100%"><tr><td bgcolor="#B00A0F">
<h1><font color="#ffffff">&nbsp; Modelo Predictivo de Clasificación XGBoost — Goles Anotados (≥ 2)</font></h1>
</td></tr></table>

> **Nota:** este notebook es un **extracto** del proyecto completo, compartido como muestra del pipeline de modelado. No contiene la totalidad del código (el web scraping, la limpieza y parte de la ingeniería de características viven en otros archivos del proyecto). Los resultados de cada etapa —entrenamiento, optimización, métricas de validación y backtesting— quedaron impresos en el **output de cada celda**, por lo que pueden revisarse directamente sin necesidad de re-ejecutar el notebook.

<table width='90%'>
<tr>
<td bgcolor='#FFBA39'>

## **<font color="#000000"> Carga, estructuración del dataset y creación del Target</font>**
</td>
</tr>
</table>

In [21]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import optuna
import joblib
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             roc_auc_score, roc_curve, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)

In [22]:
# Cargar el dataset obtenido mediante Web Scraping
df = pd.read_csv('bd_liga1_Peru_entrenamiento.csv', sep=';')

# Filtrar solo partidos hasta el 27/04/2026 inclusive
df['fecha'] = pd.to_datetime(df['fecha'], format='%d/%m/%Y')
df = df[df['fecha'] <= pd.Timestamp('2026-04-27')].copy()
df['fecha'] = df['fecha'].dt.strftime('%d/%m/%Y')

# Definir las estadísticas base (las métricas que tienen sufijo _local y _visitante)
columnas_estadisticas = [
    'goles', 'Posesión de pelota', 'Goles esperados (xG)', 'Tiros totales', 
    'Tiros a puerta', 
    'Tiros adentro del area', 'Tiros desde fuera del area', 
    'Pases', 'Pases precisos', 
    'Pases al ultimo tercio', 
    'Entradas', 'Intercepciones', 'Recuperaciones', 
    'Corners', 'Faltas',
    'Tarjetas amarillas', 'Tarjetas rojas'
]

# Desdoblar los datos para los equipos LOCALES
df_local = df[['fecha', 'equipo_local', 'equipo_visitante']].copy()
df_local.columns = ['Fecha', 'Equipo', 'Rival']
df_local['Local'] = 1

for col in columnas_estadisticas:
    df_local[col] = df[f'{col}_local']

# Desdoblar los datos para los equipos VISITANTES
df_visitante = df[['fecha', 'equipo_visitante', 'equipo_local']].copy()
df_visitante.columns = ['Fecha', 'Equipo', 'Rival']
df_visitante['Local'] = 0

for col in columnas_estadisticas:
    df_visitante[col] = df[f'{col}_visitante']

# Unir ambos sub-conjuntos en un solo dataset (formato vertical/pivotado)
df_estructurado = pd.concat([df_local, df_visitante], ignore_index=True)

# Convertir la fecha a formato de tiempo real y ordenar cronológicamente por equipo
df_estructurado['Fecha'] = pd.to_datetime(df_estructurado['Fecha'], format='%d/%m/%Y')
df_estructurado = df_estructurado.sort_values(by=['Equipo', 'Fecha']).reset_index(drop=True)

# Crear el Target
condicion_target = (df_estructurado['goles'] >= 2)
df_estructurado['Target'] = np.where(condicion_target, 1, 0)

# Verificar el resultado final
print('Dimensiones del nuevo dataset:', df_estructurado.shape)
print(f'Rango de fechas: {df_estructurado["Fecha"].min().date()} -> {df_estructurado["Fecha"].max().date()}')
print()
print(df_estructurado[['Fecha', 'Equipo', 'Rival', 'Local', 'Goles esperados (xG)', 'Tiros a puerta', 'goles', 'Target']].head(5))

Dimensiones del nuevo dataset: (2156, 22)
Rango de fechas: 2023-02-04 -> 2026-04-27

       Fecha             Equipo           Rival  Local  Goles esperados (xG)  \
0 2025-02-09  ADC Juan Pablo II      Sport Boys      0                  1.21   
1 2025-02-16  ADC Juan Pablo II  Sport Huancayo      1                  0.55   
2 2025-02-21  ADC Juan Pablo II    Alianza Lima      0                  1.13   
3 2025-03-10  ADC Juan Pablo II      Binacional      0                  0.45   
4 2025-03-30  ADC Juan Pablo II             UTC      1                  3.56   

   Tiros a puerta  goles  Target  
0               1      0       0  
1               1      0       0  
2               4      0       0  
3               2      0       0  
4               6      4       1  


<table width='90%'>
<tr>
<td bgcolor='#FFBA39'>

## **<font color="#000000"> Ratios de eficiencia y ventanas móviles</font>**
</td>
</tr>
</table>

In [ ]:
# Crear ratios de eficiencia
print("\n" + "=" * 60)
print("Creación de ratios de eficiencia")
print("=" * 60)

# precision_pases → resuelve Pases/Pases precisos 
df_estructurado['precision_pases'] = (
    df_estructurado['Pases precisos'] /
    df_estructurado['Pases'].replace(0, np.nan)
)
# precision_tiros → información más densa que Tiros totales
df_estructurado['precision_tiros'] = (
    df_estructurado['Tiros a puerta'] /
    df_estructurado['Tiros totales'].replace(0, np.nan)
)
# conversion_xg → eficacia goleadora vs calidad de oportunidades
df_estructurado['conversion_xg'] = (
    df_estructurado['goles'] /
    df_estructurado['Goles esperados (xG)'].replace(0, np.nan)
)
# ratio_area → calidad posicional del ataque
df_estructurado['ratio_area'] = (
    df_estructurado['Tiros adentro del area'] /
    df_estructurado['Tiros totales'].replace(0, np.nan)
)

print("Estadísticas de los ratios creados:")
print(df_estructurado[['precision_pases', 'precision_tiros',
                        'conversion_xg', 'ratio_area']].describe().round(3))

# Lista completa para rolling
# En esta version aun no utilizamos los ratios de eficiencia calculados
cols_para_rolling = columnas_estadisticas
print(f"\nTotal variables para rolling: {len(cols_para_rolling)}")

# Ventanas móviles combinadas 3+5
print("\n" + "=" * 60)
print("Cálculo de ventanas móviles combinadas 3+5")
print("=" * 60)

df_combo = df_estructurado.copy()

for col in cols_para_rolling:
    # Ventana 3 — momentum inmediato
    df_combo[f'{col}_prom_3'] = df_combo.groupby('Equipo')[col].transform(
        lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
    )
    # Ventana 5 — tendencia reciente
    df_combo[f'{col}_prom_5'] = df_combo.groupby('Equipo')[col].transform(
        lambda x: x.shift(1).rolling(window=5, min_periods=1).mean()
    )

df_modelado_completo = df_combo.dropna().reset_index(drop=True)

print(f"  Observaciones: {len(df_modelado_completo)}")
print(f"  Columnas totales: {df_modelado_completo.shape[1]}")


Creación de ratios de eficiencia
Estadísticas de los ratios creados:
       precision_pases  precision_tiros  conversion_xg  ratio_area
count         2152.000         2155.000       2149.000    2155.000
mean             0.768            0.341          1.047       0.562
std              0.078            0.159          1.117       0.177
min              0.456            0.000          0.000       0.000
25%              0.725            0.231          0.000       0.444
50%              0.777            0.333          0.893       0.571
75%              0.824            0.438          1.493       0.688
max              0.944            1.000         16.667       1.000

Total variables para rolling: 17

Cálculo de ventanas móviles combinadas 3+5
  Observaciones: 2126
  Columnas totales: 60


In [24]:
# Dataset final para el modelo
print("\n" + "=" * 60)
print("Dataset final")
print("=" * 60)

# Como se ve, aun no se utilizan los ratios
cols_modelo_final = cols_para_rolling

features_final = (
    [f'{col}_prom_3' for col in cols_modelo_final] +
    [f'{col}_prom_5' for col in cols_modelo_final] +
    ['Local']
)

X = df_modelado_completo[features_final]
y = df_modelado_completo['Target']

print(f"  Observaciones:    {len(df_modelado_completo)}")
print(f"  Features finales: {len(features_final)}")
print(f"  Nulos en X:       {X.isnull().sum().sum()}")
print(f"\nDistribución del Target:")
print(y.value_counts().to_string())
print(f"\nFeatures finales ({len(features_final)}):")
for i, f in enumerate(features_final, 1):
    print(f"  {i:2}. {f}")


Dataset final
  Observaciones:    2126
  Features finales: 35
  Nulos en X:       0

Distribución del Target:
Target
0    1335
1     791

Features finales (35):
   1. goles_prom_3
   2. Posesión de pelota_prom_3
   3. Goles esperados (xG)_prom_3
   4. Tiros totales_prom_3
   5. Tiros a puerta_prom_3
   6. Tiros adentro del area_prom_3
   7. Tiros desde fuera del area_prom_3
   8. Pases_prom_3
   9. Pases precisos_prom_3
  10. Pases al ultimo tercio_prom_3
  11. Entradas_prom_3
  12. Intercepciones_prom_3
  13. Recuperaciones_prom_3
  14. Corners_prom_3
  15. Faltas_prom_3
  16. Tarjetas amarillas_prom_3
  17. Tarjetas rojas_prom_3
  18. goles_prom_5
  19. Posesión de pelota_prom_5
  20. Goles esperados (xG)_prom_5
  21. Tiros totales_prom_5
  22. Tiros a puerta_prom_5
  23. Tiros adentro del area_prom_5
  24. Tiros desde fuera del area_prom_5
  25. Pases_prom_5
  26. Pases precisos_prom_5
  27. Pases al ultimo tercio_prom_5
  28. Entradas_prom_5
  29. Intercepciones_prom_5
  30. Recup

<table width='90%'>
<tr>
<td bgcolor='#FFBA39'>

## **<font color="#000000">Entrenamiento, optimización y validación del modelo XGBoost</font>**
</td>
</tr>
</table>

Este bloque es la versión que ya no vuelve a ejecutar la búsqueda de hiperparámetros con Optuna. En su lugar, carga directamente el modelo XGBoost ya entrenado cuyos hiperparámetros fueron determinados y fijados en la optimización previa. 

In [25]:
print("=" * 60)
print("DATASET FINAL — recibido")
print("=" * 60)
print(f"Observaciones totales: {len(df_modelado_completo)}")
print(f"Total features:        {len(features_final)}")
print(f"Nulos en X:            {X.isnull().sum().sum()}")
print(f"\nDistribución del Target:")
print(y.value_counts().to_string())

# División temporal 80/20
# Se utiliza para el test el 20% de datos mas recientes
fecha_corte = df_modelado_completo['Fecha'].quantile(0.8)

train_mask = df_modelado_completo['Fecha'] <= fecha_corte
test_mask  = df_modelado_completo['Fecha'] > fecha_corte

X_train = X[train_mask]
X_test  = X[test_mask]
y_train = y[train_mask]
y_test  = y[test_mask]

print("\n" + "=" * 60)
print("4.2.1 DIVISIÓN TEMPORAL DEL DATASET")
print("=" * 60)
print(f"Fecha de corte:     {fecha_corte.date()}")
print(f"Train — desde:      {df_modelado_completo[train_mask]['Fecha'].min().date()}")
print(f"Train — hasta:      {df_modelado_completo[train_mask]['Fecha'].max().date()}")
print(f"Train — partidos:   {len(X_train)}")
print(f"Test  — desde:      {df_modelado_completo[test_mask]['Fecha'].min().date()}")
print(f"Test  — hasta:      {df_modelado_completo[test_mask]['Fecha'].max().date()}")
print(f"Test  — partidos:   {len(X_test)}")

# Balanceo de clases con SMOTE solo en train
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("\n" + "=" * 60)
print("4.2.2 BALANCEO DE CLASES CON SMOTE")
print("=" * 60)
print("Train ANTES de SMOTE:")
print(y_train.value_counts().to_string())
print("\nTrain DESPUÉS de SMOTE (balanceado):")
print(y_train_res.value_counts().to_string())
print("\nTest — intacto (desbalance real del fútbol):")
print(y_test.value_counts().to_string())

# Optimización de hiperparámetros — Cargar configuración óptima ────────
print("\n" + "=" * 60)
print("CARGA DEL MODELO YA ENTRENADO")
print("=" * 60)

xgb_final_model = joblib.load('modelo_xgboost_liga_goles.pkl')

print("Modelo cargado exitosamente desde 'modelo_xgboost_liga_goles.pkl'")
print("\nHiperparámetros del modelo cargado:")
for k, v in xgb_final_model.get_params().items():
    if v is not None:
        print(f"  {k:<25} = {round(v, 6) if isinstance(v, float) else v}")

# Métricas de validación
y_pred = xgb_final_model.predict(X_test)
y_prob = xgb_final_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
auc      = roc_auc_score(y_test, y_prob)
f1       = f1_score(y_test, y_pred)

print("\n" + "=" * 60)
print("4.4 MÉTRICAS DE VALIDACIÓN")
print("=" * 60)
print(f"Accuracy:   {accuracy * 100:.2f}%")
print(f"AUC-ROC:    {auc:.4f}")
print(f"F1-Score:   {f1:.4f}")
print("\nReporte detallado:")
print(classification_report(
    y_test, y_pred,
    target_names=['Rendimiento Deficiente (0)',
                  'Alto Rendimiento Ofensivo (1)']
))


DATASET FINAL — recibido
Observaciones totales: 2126
Total features:        35
Nulos en X:            0

Distribución del Target:
Target
0    1335
1     791

4.2.1 DIVISIÓN TEMPORAL DEL DATASET
Fecha de corte:     2025-08-23
Train — desde:      2023-02-11
Train — hasta:      2025-08-23
Train — partidos:   1704
Test  — desde:      2025-08-24
Test  — hasta:      2026-04-27
Test  — partidos:   422

4.2.2 BALANCEO DE CLASES CON SMOTE
Train ANTES de SMOTE:
Target
0    1080
1     624

Train DESPUÉS de SMOTE (balanceado):
Target
0    1080
1    1080

Test — intacto (desbalance real del fútbol):
Target
0    255
1    167

CARGA DEL MODELO YA ENTRENADO
Modelo cargado exitosamente desde 'modelo_xgboost_liga_goles.pkl'

Hiperparámetros del modelo cargado:
  objective                 = binary:logistic
  colsample_bytree          = 0.909588
  enable_categorical        = False
  eval_metric               = logloss
  gamma                     = 0.708483
  learning_rate             = 0.048653
  max_dept

<table width='90%'>

<tr>
<td bgcolor='#FFBA39'>

## **<font color="#000000"> Backtesting: evaluación en partidos post-corte</font>**
</td>
</tr>
</table>

In [26]:
# Backtesting: evaluar predicciones en partidos post-corte
# Usa el modelo ya entrenado y compara contra resultados reales

FECHA_CORTE = pd.Timestamp('2026-04-27')

# 1. Cargar dataset
df_bt = pd.read_csv('bd_liga1_apertura_ronda_17.csv', sep=';')
df_bt['fecha'] = pd.to_datetime(df_bt['fecha'], format='%d/%m/%Y')
print('Partidos en CSV:', len(df_bt))
print('Rango:', df_bt['fecha'].min().date(), '->', df_bt['fecha'].max().date())

# 2. Reestructurar
df_bt['fecha'] = df_bt['fecha'].dt.strftime('%d/%m/%Y')

df_local_bt = df_bt[['fecha', 'equipo_local', 'equipo_visitante']].copy()
df_local_bt.columns = ['Fecha', 'Equipo', 'Rival']
df_local_bt['Local'] = 1
for col in columnas_estadisticas:
    df_local_bt[col] = df_bt[f'{col}_local']

df_vis_bt = df_bt[['fecha', 'equipo_visitante', 'equipo_local']].copy()
df_vis_bt.columns = ['Fecha', 'Equipo', 'Rival']
df_vis_bt['Local'] = 0
for col in columnas_estadisticas:
    df_vis_bt[col] = df_bt[f'{col}_visitante']

df_bt_struct = pd.concat([df_local_bt, df_vis_bt], ignore_index=True)
df_bt_struct['Fecha'] = pd.to_datetime(df_bt_struct['Fecha'], format='%d/%m/%Y')
df_bt_struct = df_bt_struct.sort_values(['Equipo', 'Fecha']).reset_index(drop=True)
df_bt_struct['Target'] = np.where(df_bt_struct['goles'] >= 2, 1, 0)

# 4. Rolling averages para todos los partidos
df_bt_combo = df_bt_struct.copy()
for col in cols_modelo_final:
    df_bt_combo[f'{col}_prom_3'] = df_bt_combo.groupby('Equipo')[col].transform(
        lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
    )
    df_bt_combo[f'{col}_prom_5'] = df_bt_combo.groupby('Equipo')[col].transform(
        lambda x: x.shift(1).rolling(window=5, min_periods=1).mean()
    )

df_bt_final = df_bt_combo.dropna().reset_index(drop=True)

# 5. Filtrar solo partidos POST-CORTE
df_post = df_bt_final[df_bt_final['Fecha'] > FECHA_CORTE].copy().reset_index(drop=True)
print('Partidos post-corte:', len(df_post))
print('Periodo:', df_post['Fecha'].min().date(), '->', df_post['Fecha'].max().date())

# 6. Cargar el modelo desplegado en producción
xgb_final_model = joblib.load('modelo_xgboost_liga_goles.pkl')

X_post      = df_post[features_final]
y_pred_post = xgb_final_model.predict(X_post)
y_prob_post = xgb_final_model.predict_proba(X_post)[:, 1]
y_true_post = df_post['Target']

df_post = df_post.copy()
df_post['Pred']        = y_pred_post
df_post['Prob_%']      = (y_prob_post * 100).round(1)
df_post['Target_real'] = y_true_post.values
df_post['Acierto']     = (y_pred_post == y_true_post.values)

# 7. Agrupar fechas en rondas
fechas_unicas = sorted(df_post['Fecha'].unique())
rondas_map = {}
ronda_num  = 13
grupo      = [fechas_unicas[0]]
for i in range(1, len(fechas_unicas)):
    if (fechas_unicas[i] - fechas_unicas[i-1]).days <= 3:
        grupo.append(fechas_unicas[i])
    else:
        for f in grupo:
            rondas_map[f] = ronda_num
        ronda_num += 1
        grupo = [fechas_unicas[i]]
for f in grupo:
    rondas_map[f] = ronda_num

df_post['Ronda'] = df_post['Fecha'].map(rondas_map)

# Mostrar resultados por ronda
for ronda in sorted(df_post['Ronda'].unique()):
    df_r      = df_post[df_post['Ronda'] == ronda]
    aciertos  = int(df_r['Acierto'].sum())
    total     = len(df_r)
    fecha_str = df_r['Fecha'].min().strftime('%d/%m/%Y')
    print('' + '='*65)
    print(f'  Apertura Ronda {ronda}  ({fecha_str})   {aciertos}/{total} aciertos = {aciertos/total*100:.0f}%')
    print('='*65)
    print(f"  {'Equipo':<26} {'Prob%':>6}  {'Pred':>5}  {'Real':>5}  ")
    print('  ' + '-'*58)
    for _, row in df_r.iterrows():
        pred_lbl = 'Alto' if row['Pred'] == 1 else 'Bajo'
        real_lbl = 'Alto' if row['Target_real'] == 1 else 'Bajo'
        marca    = 'OK' if row['Acierto'] else 'XX'
        print(f"  {row['Equipo']:<26} {row['Prob_%']:>6.1f}  {pred_lbl:>5}  {real_lbl:>5}  {marca}")

# Resumen global
print('' + '='*65)
print('  RESUMEN GLOBAL - Backtesting')
print('='*65)
print(f'  Partidos evaluados: {len(df_post)}')
print(f'  Accuracy:           {accuracy_score(y_true_post, y_pred_post)*100:.2f}%')
print('Accuracy por ronda:')
for ronda in sorted(df_post['Ronda'].unique()):
    df_r = df_post[df_post['Ronda'] == ronda]
    print(f'    Ronda {ronda}: {df_r["Acierto"].mean()*100:.0f}%  ({int(df_r["Acierto"].sum())}/{len(df_r)})')


Partidos en CSV: 1123
Rango: 2023-02-04 -> 2026-05-31
Partidos post-corte: 90
Periodo: 2026-05-02 -> 2026-05-31
  Apertura Ronda 13  (02/05/2026)   11/18 aciertos = 61%
  Equipo                      Prob%   Pred   Real  
  ----------------------------------------------------------
  ADC Juan Pablo II            23.0   Bajo   Bajo  OK
  ADT                          63.8   Alto   Bajo  XX
  Alianza Atlético             44.2   Bajo   Alto  XX
  Alianza Lima                 72.8   Alto   Alto  OK
  Atlético Grau                28.7   Bajo   Bajo  OK
  Cajamarca                     8.5   Bajo   Alto  XX
  Cienciano                    36.1   Bajo   Bajo  OK
  Comerciantes Unidos          20.5   Bajo   Bajo  OK
  Cusco                        57.9   Alto   Alto  OK
  Deportivo Garcilaso          12.4   Bajo   Bajo  OK
  Los Chankas                  53.1   Alto   Bajo  XX
  Melgar                       75.7   Alto   Alto  OK
  Moquegua                     33.2   Bajo   Bajo  OK
  Sport Boys    